This notebook trains adversarial policies using Generative Adversarial Imitation Learning (GAIL) with Evolutionary Strategies to match expert behavior distributions across 16-prey and 32-prey conditions using separate discriminators for predator and prey. An attack-only subset of the video data was used.

In addition, the notebook also explores using a joint discriminator for fine-tuning ontop of the existing architecture. Starting from an already converged model preserves the stability achieved in earlier stages, while allowing the reward signal to depend on the joint state of both roles. The attack-only model were chosen as the starting point.

### GAIL - Video Predator-Prey Attack

In [ ]:
# install this cell and then restart session after so that the new versions can be picked up
from google.colab import drive
drive.mount('/content/drive')

!pip install ema-pytorch geomloss[full] pykeops ultralytics deep-sort-realtime -q
!pip install -r /content/drive/MyDrive/Master-Thesis/requirements.txt -q
!pip install "numpy==2.0.2" --force-reinstall -q

In [ ]:
# set working directory
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.chdir('/content/drive/MyDrive/Predator Prey Thesis')
sys.path.insert(0, '/content/drive/MyDrive/Predator Prey Thesis')

import numpy as np
print(np.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2.0.2


In [ ]:
# set PyTorch memory allocation for GPU efficiency
import os
os.environ["PYTORCH_HIP_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# imports
import torch
from pathlib import Path
from ema_pytorch import EMA
from datetime import datetime
import random, numpy as np, torch
from utils.sim_utils import *
from utils.eval_utils import *
from utils.train_utils import *
from utils.couzin_utils import *
from utils.vec_sim_utils import *
from utils.encoder_utils import *
from utils.dataset_utils import pad_expert_tensors, pad_rollout_tensors

# sinkhorn
from geomloss import SamplesLoss

# maximum mean discrepency
from utils.mmd_loss import MMDLoss

from models.Generator import ModularPolicy
from models.Discriminator import Discriminator
from models.JointDiscriminator import JointDiscriminator

[KeOps] Compiling cuda jit compiler engine ... OK
[pyKeOps] Compiling nvrtc binder for python ... OK


In [ ]:
# training configuration
num_generations = 3000  # training generations
gamma = 0.999           # learning rate decay factor
pretrain=True           # use BC pretrain
performance_eval = 5    # evaluate performance every n generations
num_perturbations = 64  # perturbations for ES

### prey ###
lr_prey_policy = 2e-4   # learning rate prey policy
sigma_prey = 0.1        # prey ES exploration noise
prey_dis_balance_factor = 2 # prey discriminator steps per policy step
prey_noise = 0.005      # prey discriminator input noise
lr_prey_disc = 5e-4     # learning rate prey discriminator
lambda_gp_prey = 5      # prey gradient penalty factor
prey_update_mode = {"mode": "avoid", "lambda": 0.2} # prey update mode

### predator ###
lr_pred_policy = 1e-4   # learning rate predator policy
sigma_pred = 0.08       # pred scatter for ES (exploration)
pred_dis_balance_factor = 2 # pred discriminator steps per policy step
pred_noise = 0.005      # pred discriminator input noise
lr_pred_disc = 2e-4     # learning rate pred discriminator
lambda_gp_pred = 10     # pred gradient penalty factor
pred_update_mode = {"mode": "attack", "lambda": 0.1} # pred update mode

# environment settings
height = 2160
width = 2160
prey_speed = 10
pred_speed = 10
step_size = 1.0
max_turn = 0.314
pert_steps = 100 # perturbation steps
init_steps = 500 # initial steps
max_speed_norm = 25.0
env_settings = (height, width, prey_speed, pred_speed, step_size, max_turn, pert_steps, max_speed_norm)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# seeds for reproducibility
import random
import numpy as np
import torch
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# load expert tensors
window_path = Path(f'/content/drive/MyDrive/Predator Prey Thesis/data/1. Data Processing/Processed/video/expert_tensors/windows/10 windows (split by attack or interaction -- used for calculating speed)/attack')

print(sorted(os.listdir(window_path)))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

['pred_tensors_hl_attack_w10_n2200.pkl', 'prey_tensors_hl_attack_w10_n2200.pkl']


In [ ]:
# load paired predator and prey expert trajectory files
def load_expert_pairs(window_path, device=None):
    files = sorted(os.listdir(window_path))
    preds = [f for f in files if f.startswith('pred')]
    preys = [f for f in files if f.startswith('prey')]

    # match files by extracting key from filename
    key = lambda name: name.split('_', 1)[1] # strip the pred/prey prefix
    pred_map = {key(f): f for f in preds}
    prey_map = {key(f): f for f in preys}

    # ensure all files are paired
    unmatched = set(pred_map) ^ set(prey_map)
    if unmatched:
        raise RuntimeError(f"unpaired expert files: {sorted(unmatched)}")

    # load and concatenate
    pred_list, prey_list = [], []
    for k in sorted(pred_map):
        pd = torch.load(os.path.join(window_path, pred_map[k]), weights_only=False)
        py = torch.load(os.path.join(window_path, prey_map[k]), weights_only=False)
        assert pd.shape[0] == py.shape[0], f"{k}: {pd.shape[0]} pred vs {py.shape[0]} prey"
        pred_list.append(pd); prey_list.append(py)

    p = torch.cat(pred_list, 0).type(torch.float32)
    q = torch.cat(prey_list, 0).type(torch.float32)
    return (p.to(device), q.to(device)) if device else (p, q)

# load expert trajectories
exp_pred_tensor, exp_prey_tensor = load_expert_pairs(window_path, device)
print("Pred Tensor Shape:", exp_pred_tensor.shape)
print("Prey Tensor Shape:", exp_prey_tensor.shape)

Pred Tensor Shape: torch.Size([2200, 10, 1, 32, 6])
Prey Tensor Shape: torch.Size([2200, 10, 32, 32, 7])


In [ ]:
# filter by prey count
active_check = exp_prey_tensor[:, 0, :, 0, -2] # (N, 32): frame 0, all agents, neighbor 0, active flag
real_agent_mask = (active_check > 0) # (N, 32) boolean mask
n_prey_per_window = real_agent_mask.sum(dim=1) # (N,) count per window

# seperate into 16 and 32 prey datasets
cond16 = (n_prey_per_window == 16)
cond32 = (n_prey_per_window == 32)

exp_prey_tensor_16 = exp_prey_tensor[cond16]
exp_prey_tensor_32 = exp_prey_tensor[cond32]
exp_pred_tensor_16 = exp_pred_tensor[cond16]
exp_pred_tensor_32 = exp_pred_tensor[cond32]

print(f"16-prey windows: {cond16.sum().item()}  |  32-prey windows: {cond32.sum().item()}")

16-prey windows: 1431  |  32-prey windows: 769


In [ ]:
# load both init pools
init_pool_path_16 = Path('/content/drive/MyDrive/Predator Prey Thesis/data/1. Data Processing/Processed/init_pool/init_pool_16prey.pt')
init_pool_16 = torch.load(init_pool_path_16).to(device)

init_pool_path_32 = Path('/content/drive/MyDrive/Predator Prey Thesis/data/1. Data Processing/Processed/init_pool/init_pool_32prey.pt')
init_pool_32 = torch.load(init_pool_path_32).to(device)

In [ ]:
# load and freeze encoders

# initlize prey encoder and load weights from pretraining
prey_encoder = TransitionEncoder(features=6, embd_dim=32, z=32).to(device) # was 5

prey_encoder_path = Path('/content/drive/MyDrive/Predator Prey Thesis/models/trained_policies/Encoder/prey_encoder_attack.pt')
prey_state = torch.load(prey_encoder_path, map_location=device)
prey_encoder.load_state_dict(prey_state)

# freeze prey encoder
for p in prey_encoder.parameters():
    p.requires_grad = False
prey_encoder.eval()
print("Prey Encoder loaded & frozen.\n")

# initialize predator encoder and load weights from pretraining
pred_encoder = TransitionEncoder(features=5, embd_dim=32, z=32).to(device) # was 4
pred_encoder_path = Path('/content/drive/MyDrive/Predator Prey Thesis/models/trained_policies/Encoder/pred_encoder_attack.pt')
pred_state = torch.load(pred_encoder_path, map_location=device)
pred_encoder.load_state_dict(pred_state)

# freeze predator encoder
for p in pred_encoder.parameters():
    p.requires_grad = False
pred_encoder.eval()
print("Predator Encoder loaded & frozen.")

Prey Encoder loaded & frozen.

Predator Encoder loaded & frozen.


In [ ]:
# load pretrained policies (behavioral cloning)

# initialize prey policy and load pretrained weights
prey_policy = ModularPolicy(features=6).to(device) # was 5
prey_policy.set_parameters()

bc_prey_path = Path('/content/drive/MyDrive/Predator Prey Thesis/models/trained_policies/BC-Policy/bc_prey_policy_attack.pt')

if pretrain:
    prey_policy.load_state_dict(torch.load(bc_prey_path, map_location=device))

# initialize EMA for prey policy
ema_prey = EMA(prey_policy, beta=0.9999, update_after_step=10, update_every=5, allow_different_devices=True)
print("Pretrained Prey Policy loaded.\n")

Pretrained Prey Policy loaded.



In [ ]:
# initialize predator policy and load pretrained weights
pred_policy = ModularPolicy(features=5).to(device) # was 4
pred_policy.set_parameters()

bc_pred_path = Path('/content/drive/MyDrive/Predator Prey Thesis/models/trained_policies/BC-Policy/bc_pred_policy_attack.pt')


if pretrain:
    pred_policy.load_state_dict(torch.load(bc_pred_path, map_location=device))

# initialize EMA for predator policy
ema_pred = EMA(pred_policy, beta=0.9999, update_after_step=10, update_every=5, allow_different_devices=True)
print("Pretrained Predator Policy loaded.\n")

Pretrained Predator Policy loaded.



In [ ]:
# store initial parameter norms to prevent ES updates from drifting too far
theta_norm_ref_pred_pin = nn.utils.parameters_to_vector(pred_policy.pairwise.parameters()).norm().item()
theta_norm_ref_pred_an  = nn.utils.parameters_to_vector(pred_policy.attention.parameters()).norm().item()
theta_norm_ref_prey_pin = nn.utils.parameters_to_vector(prey_policy.pairwise.parameters()).norm().item()
theta_norm_ref_prey_an  = nn.utils.parameters_to_vector(prey_policy.attention.parameters()).norm().item()

In [ ]:
# initialize prey discriminator
prey_discriminator = Discriminator(encoder=prey_encoder, role="prey", z_dim=32).to(device)
prey_discriminator.set_parameters(init=True)
optim_disc_prey = torch.optim.RMSprop(prey_discriminator.parameters(), lr=lr_prey_disc, alpha=0.99, eps=1e-08)

# initialize predator discriminator
pred_discriminator = Discriminator(encoder=pred_encoder, role="predator", z_dim=32).to(device)
pred_discriminator.set_parameters(init=True)
optim_disc_pred = torch.optim.RMSprop(pred_discriminator.parameters(), lr=lr_pred_disc, alpha=0.99, eps=1e-08)

# initialize losses
prey_mmd_loss = MMDLoss(encoder=prey_encoder, role="prey").to(device)
pred_mmd_loss = MMDLoss(encoder=pred_encoder, role="predator").to(device)
sinkhorn_loss = SamplesLoss(loss="sinkhorn", backend="tensorized")

In [ ]:
# get expert distribution statistics for 32 prey
mmd_means, mmd_stds, sinkhorn_means, sinkhorn_stds = get_expert_values(
    exp_pred_tensor=exp_pred_tensor_32, exp_prey_tensor=exp_prey_tensor_32,
    prey_mmd_loss=prey_mmd_loss, pred_mmd_loss=pred_mmd_loss,
    prey_encoder=prey_encoder, pred_encoder=pred_encoder,
    sinkhorn_loss=sinkhorn_loss)

MMD_PREY, MMD_PRED = float(mmd_means[0]), float(mmd_means[1])
SNK_PREY, SNK_PRED = float(sinkhorn_means[0]), float(sinkhorn_means[1])

  0%|          | 0/500 [00:00<?, ?it/s]


Expert Prey MMD: 0.04253654885292053 ± 0.029013924397851748
Expert Prey Sinkhorn: 6.385139579288079e-05 ± 1.4364113360817974e-05

Expert Pred MMD: 0.26044760608673095 ± 0.1306330089689001
Expert Pred Sinkhorn: 0.00022810194623889402 ± 5.0894882754735965e-05


In [ ]:
# get expert distribution statistics for 16 prey
mmd_means16, mmd_stds16, sinkhorn_means16, sinkhorn_stds16 = get_expert_values(
    exp_pred_tensor=exp_pred_tensor_16, exp_prey_tensor=exp_prey_tensor_16,
    prey_mmd_loss=prey_mmd_loss, pred_mmd_loss=pred_mmd_loss,
    prey_encoder=prey_encoder, pred_encoder=pred_encoder,
    sinkhorn_loss=sinkhorn_loss)

MMD_PREY16, MMD_PRED16 = float(mmd_means16[0]), float(mmd_means16[1])
SNK_PREY16, SNK_PRED16 = float(sinkhorn_means16[0]), float(sinkhorn_means16[1])

  0%|          | 0/500 [00:00<?, ?it/s]


Expert Prey MMD: 0.005902616500854492 ± 0.0044028656323427265
Expert Prey Sinkhorn: 2.4959040679277676e-06 ± 7.99665326405307e-07

Expert Pred MMD: 0.3377178223133087 ± 0.1704688422997123
Expert Pred Sinkhorn: 9.230560110154329e-05 ± 2.3786692703446604e-05


In [ ]:
# training setup
metrics_list = []
metrics_list_16 = []
policy_metrics_list = []
disc_metrics_list = []
gamma = 0.999

best_prey = float("inf")
best_pred = float("inf")
best_prey_policy_state = {k: v.detach().clone() for k, v in ema_prey.ema_model.state_dict().items()}
best_pred_policy_state = {k: v.detach().clone() for k, v in ema_pred.ema_model.state_dict().items()}

run_dir = Path('/content/drive/MyDrive/Predator Prey Thesis/data/2. Training/VideoPredPrey - GAIL/stage1_seed42_32and16_attackonly')
run_dir.mkdir(parents=True, exist_ok=True)

# training Loop
for gen in range(num_generations):

    # select condition: 50/50 between 16-prey and 32-prey
    n_prey = random.choice([16, 32])
    if n_prey == 16:
        cond_init_pool = init_pool_16
        cond_exp_pred_tensor = exp_pred_tensor_16
        cond_exp_prey_tensor = exp_prey_tensor_16
    else:
        cond_init_pool = init_pool_32
        cond_exp_pred_tensor = exp_pred_tensor_32
        cond_exp_prey_tensor = exp_prey_tensor_32

    # use EMA models for rollouts
    rollout_prey_policy = ema_prey.ema_model.to(device)
    rollout_pred_policy = ema_pred.ema_model.to(device)

    # generate rollouts with current policies
    gen_pred_tensor, gen_prey_tensor = run_env_vectorized(prey_policy=rollout_prey_policy, pred_policy=rollout_pred_policy,
                                                          n_prey=n_prey, n_pred=1, max_steps=100, init_pool=cond_init_pool,
                                                          area_width=width, area_height=height,
                                                          prey_speed=prey_speed, pred_speed=pred_speed,
                                                          step_size=step_size, max_turn=max_turn,
                                                          max_speed_norm=max_speed_norm)

    # pad to max_prey=32 format (adds active-mask column)
    gen_pred_tensor, gen_prey_tensor = pad_rollout_tensors(gen_pred_tensor, gen_prey_tensor, max_prey=32)

    # update discriminators
    for i in range(pred_dis_balance_factor):
        expert_pred_batch, _ = sample_data(cond_exp_pred_tensor, batch_size=20, window_len=10)
        generative_pred_batch, _ = sample_data(gen_pred_tensor, batch_size=20, window_len=10)
        expert_pred_batch = expert_pred_batch.to(device)
        generative_pred_batch = generative_pred_batch.to(device)
        dis_metric_pred = pred_discriminator.update(expert_pred_batch, generative_pred_batch, optim_disc_pred,
                                                    lambda_gp_pred, noise=pred_noise, generation=gen, num_generations=num_generations)

    for i in range(prey_dis_balance_factor):
        expert_prey_batch, _ = sample_data(cond_exp_prey_tensor, batch_size=10, window_len=10)
        generative_prey_batch, _ = sample_data(gen_prey_tensor, batch_size=10, window_len=10)
        expert_prey_batch = expert_prey_batch.to(device)
        generative_prey_batch = generative_prey_batch.to(device)
        dis_metric_prey = prey_discriminator.update(expert_prey_batch, generative_prey_batch, optim_disc_prey,
                                                    lambda_gp_prey, noise=prey_noise, generation=gen, num_generations=num_generations)

        disc_metrics_list.append((dis_metric_prey, dis_metric_pred))


    # sample initial positions for ES
    init_pos = init_positions(cond_init_pool, batch=num_perturbations, mode="dual",
                          area_width=width, area_height=height)

    # optimize pred PIN with ES
    pin_pred_metrics = optimize_es(pred_policy=pred_policy, prey_policy=prey_policy,
                               role="pred", module="pairwise", mode=pred_update_mode,
                               discriminator=pred_discriminator, lr=lr_pred_policy, sigma=sigma_pred,
                               num_perturbations=num_perturbations, init_pos=init_pos,
                               settings_batch_env=env_settings, n_prey=n_prey,
                               theta_norm_ref=theta_norm_ref_pred_pin)

    # optimize pred AN with ES
    an_pred_metrics  = optimize_es(pred_policy=pred_policy, prey_policy=prey_policy,
                               role="pred", module="attention", mode=pred_update_mode,
                               discriminator=pred_discriminator, lr=lr_pred_policy, sigma=sigma_pred,
                               num_perturbations=num_perturbations, init_pos=init_pos,
                               settings_batch_env=env_settings, n_prey=n_prey,
                               theta_norm_ref=theta_norm_ref_pred_an)
    ema_pred.update()

    # optimize prey PIN with ES
    pin_prey_metrics = optimize_es(pred_policy=pred_policy, prey_policy=prey_policy,
                               role="prey", module="pairwise", mode=prey_update_mode,
                               discriminator=prey_discriminator, lr=lr_prey_policy, sigma=sigma_prey,
                               num_perturbations=num_perturbations, init_pos=init_pos,
                               settings_batch_env=env_settings, n_prey=n_prey,
                               theta_norm_ref=theta_norm_ref_prey_pin)

    # optimize prey AN with ES
    an_prey_metrics  = optimize_es(pred_policy=pred_policy, prey_policy=prey_policy,
                               role="prey", module="attention", mode=prey_update_mode,
                               discriminator=prey_discriminator, lr=lr_prey_policy, sigma=sigma_prey,
                               num_perturbations=num_perturbations, init_pos=init_pos,
                               settings_batch_env=env_settings, n_prey=n_prey,
                               theta_norm_ref=theta_norm_ref_prey_an)
    ema_prey.update()

    # memory clean up
    import gc
    gc.collect()
    torch.cuda.empty_cache()

    # append metrics
    policy_metrics_list.append({"predator": (pin_pred_metrics, an_pred_metrics), "prey": (pin_prey_metrics, an_prey_metrics), "n_prey": n_prey})

    # decay learning rates and exploration noise
    lr_pred_policy *= gamma
    lr_prey_policy *= gamma
    sigma_pred *= gamma
    sigma_prey *= gamma

    # evalute current policies
    if gen % performance_eval == 0:
        m32 = calculate_metrics(pred_policy=pred_policy, prey_policy=prey_policy,
                                prey_encoder=prey_encoder, pred_encoder=pred_encoder,
                                exp_prey_tensor=exp_prey_tensor_32, exp_pred_tensor=exp_pred_tensor_32,
                                prey_mmd_loss=prey_mmd_loss, pred_mmd_loss=pred_mmd_loss,
                                sinkhorn_loss=sinkhorn_loss,
                                init_pool=init_pool_32, n_prey=32, env_settings=env_settings, device="cuda")
        m16 = calculate_metrics(pred_policy=pred_policy, prey_policy=prey_policy,
                                prey_encoder=prey_encoder, pred_encoder=pred_encoder,
                                exp_prey_tensor=exp_prey_tensor_16, exp_pred_tensor=exp_pred_tensor_16,
                                prey_mmd_loss=prey_mmd_loss, pred_mmd_loss=pred_mmd_loss,
                                sinkhorn_loss=sinkhorn_loss,
                                init_pool=init_pool_16, n_prey=16, env_settings=env_settings, device="cuda")

        metrics = m32
        metrics_list.append(m32)
        metrics_list_16.append({"gen": gen, **m16})

        current_prey_state = ema_prey.ema_model.state_dict()
        current_pred_state = ema_pred.ema_model.state_dict()

        # selection on 32 only
        prey_score = m32["mmd_prey_mean"]/MMD_PREY + m32["sinkhorn_prey_mean"]/SNK_PREY
        pred_score = m32["mmd_pred_mean"]/MMD_PRED + m32["sinkhorn_pred_mean"]/SNK_PRED

        if prey_score < best_prey:
            best_prey = prey_score
            best_prey_policy_state = {k: v.detach().clone() for k, v in current_prey_state.items()}
            print(f"New best PREY {prey_score:.2f}x  "
                  f"(mmd {m32['mmd_prey_mean']/MMD_PREY:.1f}x, snk {m32['sinkhorn_prey_mean']/SNK_PREY:.1f}x)\n")
        if pred_score < best_pred:
            best_pred = pred_score
            best_pred_policy_state = {k: v.detach().clone() for k, v in current_pred_state.items()}
            print(f"New best PRED {pred_score:.2f}x  "
                  f"(mmd {m32['mmd_pred_mean']/MMD_PRED:.1f}x, snk {m32['sinkhorn_pred_mean']/SNK_PRED:.1f}x)\n")

        print(f"[16-PREY] prey {m16['mmd_prey_mean']/MMD_PREY16:.2f}x mmd, "
              f"{m16['sinkhorn_prey_mean']/SNK_PREY16:.2f}x snk | "
              f"pred {m16['mmd_pred_mean']/MMD_PRED16:.2f}x mmd, "
              f"{m16['sinkhorn_pred_mean']/SNK_PRED16:.2f}x snk")

    print(f"Generation {gen+1}  [condition: n_prey={n_prey}]")
    print(f"[PREY] PIN Network:   {pin_prey_metrics}")
    print(f"[PREY] AN Network:    {an_prey_metrics}")
    print(f"[PREY] Discriminator: {dis_metric_prey}")
    print(f"[PREY] Score Diff: {abs(dis_metric_prey['expert_score_mean'] - dis_metric_prey['policy_score_mean'])}")
    if gen % performance_eval == 0:
        print(f"[PREY] MMD: {metrics['mmd_prey_mean']:.4f} ± {metrics['mmd_prey_std']:.4f} | "
              f"Sinkhorn: {metrics['sinkhorn_prey_mean']:.4f} ± {metrics['sinkhorn_prey_std']:.4f}")
    print("--------------------------------")
    print(f"[PRED] PIN Network:   {pin_pred_metrics}")
    print(f"[PRED] AN Network:    {an_pred_metrics}")
    print(f"[PRED] Discriminator: {dis_metric_pred}")
    print(f"[PRED] Score Diff: {abs(dis_metric_pred['expert_score_mean'] - dis_metric_pred['policy_score_mean'])}")
    if gen % performance_eval == 0:
        print(f"[PRED] MMD: {metrics['mmd_pred_mean']:.4f} ± {metrics['mmd_pred_std']:.4f} | "
              f"Sinkhorn: {metrics['sinkhorn_pred_mean']:.4f} ± {metrics['sinkhorn_pred_std']:.4f}\n")

    # save checkpoint every 50 generations
    if (gen + 1) % 50 == 0:
        torch.save({
            "gen": gen,
            "prey_policy": prey_policy.state_dict(), "pred_policy": pred_policy.state_dict(),
            "ema_prey": ema_prey.state_dict(), "ema_pred": ema_pred.state_dict(),
            "prey_disc": prey_discriminator.state_dict(), "pred_disc": pred_discriminator.state_dict(),
            "optim_disc_prey": optim_disc_prey.state_dict(), "optim_disc_pred": optim_disc_pred.state_dict(),
            "best_prey_policy_state": best_prey_policy_state, "best_pred_policy_state": best_pred_policy_state,
            "best_prey": best_prey, "best_pred": best_pred,
            "lr_pred_policy": lr_pred_policy, "lr_prey_policy": lr_prey_policy,
            "sigma_pred": sigma_pred, "sigma_prey": sigma_prey,
            "metrics_list": metrics_list, "metrics_list_16": metrics_list_16,
            "policy_metrics_list": policy_metrics_list, "disc_metrics_list": disc_metrics_list,
        }, run_dir / "ckpt_latest.pt")

# load best policies
prey_policy.load_state_dict(best_prey_policy_state, strict=True)
pred_policy.load_state_dict(best_pred_policy_state, strict=True)

In [ ]:
# save trained policies
run_dir = Path('/content/drive/MyDrive/Predator Prey Thesis/data/2. Training/VideoPredPrey - GAIL/stage1_seed42_32and16_attackonly')
run_dir.mkdir(parents=True, exist_ok=True)

prey_path = run_dir / "prey_policy_stage1.pth"
torch.save(prey_policy.state_dict(), prey_path)

pred_path = run_dir / "pred_policy_stage1.pth"
torch.save(pred_policy.state_dict(), pred_path)

In [ ]:
# plot trianing metrics

%matplotlib inline
# plot training and evaluation metrics for prey
plot_train_metrics(disc_metrics_list, prey_dis_balance_factor, role="prey", save_dir=run_dir)
plot_es_metrics(policy_metrics_list, role="prey", save_dir=run_dir)
plot_eval_metrics(metrics_list, role="prey",
                  mmd_means=mmd_means, mmd_stds=mmd_stds,
                  sinkhorn_means=sinkhorn_means, sinkhorn_stds=sinkhorn_stds,
                  max_steps=num_generations, save_dir=run_dir)

In [ ]:
%matplotlib inline
# plot training and evaluation metrics for predator
plot_train_metrics(disc_metrics_list, pred_dis_balance_factor, role="predator", save_dir=run_dir)
plot_es_metrics(policy_metrics_list, role="predator", save_dir=run_dir)
plot_eval_metrics(metrics_list, role="predator",
                  mmd_means=mmd_means, mmd_stds=mmd_stds,
                  sinkhorn_means=sinkhorn_means, sinkhorn_stds=sinkhorn_stds,
                  max_steps=num_generations, save_dir=run_dir)

In [ ]:
# visualize
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

# final evaluation metrics
for tag, ip, ep, epy, n in [("32", init_pool_32, exp_pred_tensor_32, exp_prey_tensor_32, 32),
                            ("16", init_pool_16, exp_pred_tensor_16, exp_prey_tensor_16, 16)]:
    fm = calculate_metrics(pred_policy=pred_policy, prey_policy=prey_policy,
                           prey_encoder=prey_encoder, pred_encoder=pred_encoder,
                           exp_prey_tensor=epy, exp_pred_tensor=ep,
                           prey_mmd_loss=prey_mmd_loss, pred_mmd_loss=pred_mmd_loss,
                           sinkhorn_loss=sinkhorn_loss, init_pool=ip, n_prey=n,
                           env_settings=env_settings, device="cuda", n_episodes=10)

# generate and save rollout visualization
def visualise(n_prey_vis, init_pool_vis, tag):
    _, _, gm = run_env_simulation(visualization='off',
                                  prey_policy=prey_policy, pred_policy=pred_policy,
                                  n_prey=n_prey_vis, n_pred=1, max_steps=100,
                                  pred_speed=pred_speed, prey_speed=prey_speed,
                                  area_width=width, area_height=height,
                                  max_turn=max_turn, step_size=step_size,
                                  max_speed_norm=max_speed_norm, init_pool=init_pool_vis)
    steps, _ = gm

    # static frames
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    for ax, s in zip(axes.flat, np.linspace(0, len(steps) - 1, 10, dtype=int)):
        m = steps[s]
        xs, ys = m["xs"] * width, m["ys"] * height
        ax.scatter(xs[1:], ys[1:], c='black', s=10)
        ax.scatter(xs[0],  ys[0],  c='red',   s=40)
        ax.set_xlim(0, width); ax.set_ylim(0, height)
        ax.set_aspect('equal'); ax.set_title(f"{tag} prey — step {s}")
    plt.tight_layout()
    plt.show()

    # animation
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_xlim(0, width); ax.set_ylim(0, height); ax.set_aspect('equal')
    pred_scat = ax.scatter([], [], c='red',  s=60, label='Predator')
    prey_scat = ax.scatter([], [], c='blue', s=20, label='Prey')
    ax.legend(loc='upper right')

    def update(frame):
        m = steps[frame]
        xs = np.asarray(m["xs"]) * width
        ys = np.asarray(m["ys"]) * height
        pred_scat.set_offsets([[xs[0], ys[0]]])
        prey_scat.set_offsets(np.stack([xs[1:], ys[1:]], axis=1))
        ax.set_title(f"{tag} prey — step {frame}")
        return pred_scat, prey_scat

    ani = animation.FuncAnimation(fig, update, frames=len(steps), interval=1000/15, blit=True)
    out_path = run_dir / f"rollout_stage1_{tag}prey.mp4"
    ani.save(str(out_path), writer=animation.FFMpegWriter(fps=15, bitrate=1800))
    plt.close(fig)
    print(f"{tag}-prey video:", out_path.exists(), out_path.stat().st_size)
    return gm

sim_32 = visualise(32, init_pool_32, "32")
sim_16 = visualise(16, init_pool_16, "16")

### Exploring Joint Discriminator as a Stage 2

In [ ]:
# mount + imports
from google.colab import drive
drive.mount('/content/drive')
import os, sys, copy, json, random, time
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from pathlib import Path

PROJECT = '/content/drive/MyDrive/Predator Prey Thesis'
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

from ema_pytorch import EMA
from geomloss import SamplesLoss

from utils.sim_utils import *
from utils.eval_utils import *
from utils.train_utils import *
from utils.vec_sim_utils import *
from utils.encoder_utils import *
from utils.dataset_utils import pad_expert_tensors, pad_rollout_tensors
from utils.mmd_loss import MMDLoss

from models.Generator import ModularPolicy
from models.JointDiscriminator import JointDiscriminator

# seeds
random.seed(42); np.random.seed(42)
torch.manual_seed(42); torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# env settings
height, width = 2160, 2160
prey_speed, pred_speed = 10, 10
step_size = 1.0
max_turn = 0.314
pert_steps = 100
max_speed_norm = 25.0
env_settings = (height, width, prey_speed, pred_speed, step_size, max_turn, pert_steps, max_speed_norm)

num_perturbations = 64
performance_eval = 5
prey_update_mode = {"mode": "avoid",  "lambda": 0.2}
pred_update_mode = {"mode": "attack", "lambda": 0.1}

# paths
P = Path(PROJECT)
WINDOW_PATH = P / 'data/1. Data Processing/Processed/video/expert_tensors/windows/10 windows (split by attack or interaction -- used for calculating speed)/attack'
INIT_POOL_16 = P / 'data/1. Data Processing/Processed/init_pool/init_pool_16prey.pt'
INIT_POOL_32 = P / 'data/1. Data Processing/Processed/init_pool/init_pool_32prey.pt'
PREY_ENC = P / 'models/trained_policies/Encoder/prey_encoder_attack.pt'
PRED_ENC = P / 'models/trained_policies/Encoder/pred_encoder_attack.pt'
BC_PREY = P / 'models/trained_policies/BC-Policy/bc_prey_policy_attack.pt'
BC_PRED = P / 'models/trained_policies/BC-Policy/bc_pred_policy_attack.pt'

# stage 2 starts from the stage 1 attack
STAGE1_DIR = P / 'data/2. Training/VideoPredPrey - GAIL/stage1_seed42_32and16_attackonly'
SWEEP_DIR  = P / 'data/2. Training/VideoPredPrey - GAIL/stage2_seed42_jointdisc'
SWEEP_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# expert tensors
def load_expert_pairs(window_path, device = None):
    files = sorted(os.listdir(window_path))
    preds = [f for f in files if f.startswith('pred')]
    preys = [f for f in files if f.startswith('prey')]

    # match files by extracting key from filename
    key = lambda name: name.split('_', 1)[1] # strip the pred/prey prefix
    pred_map = {key(f): f for f in preds}
    prey_map = {key(f): f for f in preys}

    # ensure all files are paired
    unmatched = set(pred_map) ^ set(prey_map)
    if unmatched:
        raise RuntimeError(f"unpaired expert files: {sorted(unmatched)}")

    # load and concatenate
    pred_list, prey_list = [], []
    for k in sorted(pred_map):
        pd_ = torch.load(os.path.join(window_path, pred_map[k]), weights_only=False)
        py_ = torch.load(os.path.join(window_path, prey_map[k]), weights_only=False)
        assert pd_.shape[0] == py_.shape[0], f"{k}: {pd_.shape[0]} pred vs {py_.shape[0]} prey"
        pred_list.append(pd_); prey_list.append(py_)

    p = torch.cat(pred_list, 0).type(torch.float32)
    q = torch.cat(prey_list, 0).type(torch.float32)

    return (p.to(device), q.to(device)) if device else (p, q)

In [ ]:
# load expert trajectories
exp_pred_tensor, exp_prey_tensor = load_expert_pairs(WINDOW_PATH, device)
print("Pred Tensor:", tuple(exp_pred_tensor.shape))
print("Prey Tensor:", tuple(exp_prey_tensor.shape))

# split by prey count
active_check      = exp_prey_tensor[:, 0, :, 0, -2]
n_prey_per_window = (active_check > 0).sum(dim=1)
cond16 = (n_prey_per_window == 16)
cond32 = (n_prey_per_window == 32)

exp_prey_tensor_16 = exp_prey_tensor[cond16]
exp_prey_tensor_32 = exp_prey_tensor[cond32]
exp_pred_tensor_16 = exp_pred_tensor[cond16]
exp_pred_tensor_32 = exp_pred_tensor[cond32]
print(f"16-prey windows: {cond16.sum().item()}  |  32-prey windows: {cond32.sum().item()}")

init_pool_16 = torch.load(INIT_POOL_16).to(device)
init_pool_32 = torch.load(INIT_POOL_32).to(device)

Pred Tensor: (2200, 10, 1, 32, 6)
Prey Tensor: (2200, 10, 32, 32, 7)
16-prey windows: 1431  |  32-prey windows: 769


In [ ]:
# load and freeze encoders

# initlize prey encoder and load weights from pretraining + freeze prey encoder
prey_encoder = TransitionEncoder(features=6, embd_dim=32, z=32).to(device)
prey_encoder.load_state_dict(torch.load(PREY_ENC, map_location=device))

for p_ in prey_encoder.parameters():
  p_.requires_grad = False
prey_encoder.eval()

# initialize predator encoder and load weights from pretraining + freeze predator encoder
pred_encoder = TransitionEncoder(features=5, embd_dim=32, z=32).to(device)
pred_encoder.load_state_dict(torch.load(PRED_ENC, map_location=device))

for p_ in pred_encoder.parameters():
  p_.requires_grad = False
pred_encoder.eval()

print("Encoders loaded & frozen.")

prey_mmd_loss = MMDLoss(encoder=prey_encoder, role="prey").to(device)
pred_mmd_loss = MMDLoss(encoder=pred_encoder, role="predator").to(device)
sinkhorn_loss = SamplesLoss(loss="sinkhorn", backend="tensorized")

# expert baselines
BASELINE_JSON = SWEEP_DIR / "expert_baseline.json"
if BASELINE_JSON.exists():
    _b = json.load(open(BASELINE_JSON))
    mmd_means, mmd_stds = _b["mmd_means"], _b["mmd_stds"]
    sinkhorn_means, sinkhorn_stds = _b["sinkhorn_means"], _b["sinkhorn_stds"]
    mmd_means16, sinkhorn_means16 = _b["mmd_means16"], _b["sinkhorn_means16"]
else:
    mmd_means, mmd_stds, sinkhorn_means, sinkhorn_stds = get_expert_values(exp_pred_tensor=exp_pred_tensor_32, exp_prey_tensor=exp_prey_tensor_32, prey_mmd_loss=prey_mmd_loss, pred_mmd_loss=pred_mmd_loss, prey_encoder=prey_encoder, pred_encoder=pred_encoder, sinkhorn_loss=sinkhorn_loss)
    mmd_means16, mmd_stds16, sinkhorn_means16, sinkhorn_stds16 = get_expert_values(exp_pred_tensor=exp_pred_tensor_16, exp_prey_tensor=exp_prey_tensor_16, prey_mmd_loss=prey_mmd_loss, pred_mmd_loss=pred_mmd_loss, prey_encoder=prey_encoder, pred_encoder=pred_encoder, sinkhorn_loss=sinkhorn_loss)
    json.dump({"mmd_means": [float(x) for x in mmd_means], "mmd_stds": [float(x) for x in mmd_stds],
               "sinkhorn_means": [float(x) for x in sinkhorn_means], "sinkhorn_stds": [float(x) for x in sinkhorn_stds],
               "mmd_means16": [float(x) for x in mmd_means16], "sinkhorn_means16": [float(x) for x in sinkhorn_means16]},
              open(BASELINE_JSON, "w"), indent=2)

MMD_PREY, MMD_PRED = float(mmd_means[0]), float(mmd_means[1])
SNK_PREY, SNK_PRED = float(sinkhorn_means[0]), float(sinkhorn_means[1])
MMD_PREY16, MMD_PRED16 = float(mmd_means16[0]), float(mmd_means16[1])
SNK_PREY16, SNK_PRED16 = float(sinkhorn_means16[0]), float(sinkhorn_means16[1])

In [ ]:
# load stage 1
prey_policy_tmpl = ModularPolicy(features=6).to(device)
pred_policy_tmpl = ModularPolicy(features=5).to(device)
prey_policy_tmpl.load_state_dict(torch.load(STAGE1_DIR / "prey_policy_stage1.pth", map_location=device))
pred_policy_tmpl.load_state_dict(torch.load(STAGE1_DIR / "pred_policy_stage1.pth", map_location=device))

# ES relative-clip anchors
THETA_REF = {
    ("pred", "pairwise"):  nn.utils.parameters_to_vector(pred_policy_tmpl.pairwise.parameters()).norm().item(),
    ("pred", "attention"): nn.utils.parameters_to_vector(pred_policy_tmpl.attention.parameters()).norm().item(),
    ("prey", "pairwise"):  nn.utils.parameters_to_vector(prey_policy_tmpl.pairwise.parameters()).norm().item(),
    ("prey", "attention"): nn.utils.parameters_to_vector(prey_policy_tmpl.attention.parameters()).norm().item(),
}
print("theta_norm_ref:", {k: round(v, 4) for k, v in THETA_REF.items()})

joint_disc = JointDiscriminator(pred_encoder=pred_encoder,
                                prey_encoder=prey_encoder, z_dim=32).to(device)

In [ ]:
# joint discriminator
def reset_joint_head(disc):
    for name in ("fc1", "fc2", "fc3", "fc4"):
        getattr(disc, name).reset_parameters()

_enc_fingerprint = (prey_encoder.state_dict()[list(prey_encoder.state_dict())[0]].clone(),
                    pred_encoder.state_dict()[list(pred_encoder.state_dict())[0]].clone())

def assert_encoders_intact():
    a = prey_encoder.state_dict()[list(prey_encoder.state_dict())[0]]
    b = pred_encoder.state_dict()[list(pred_encoder.state_dict())[0]]
    assert torch.equal(a, _enc_fingerprint[0]), "PREY ENCODER WAS RE-INITIALIZED"
    assert torch.equal(b, _enc_fingerprint[1]), "PRED ENCODER WAS RE-INITIALIZED"

reset_joint_head(joint_disc)
assert_encoders_intact()
print("Joint discriminator ready; pretrained encoders intact.")

In [ ]:
# stage 2 hyperparameter sweep

def run_one_config(cfg, out_root, ckpt_every=25, resume=True):
    """Train one Stage 2 config. """
    run_id = cfg["run_id"]
    cfg_dir = Path(out_root) / run_id
    cfg_dir.mkdir(parents=True, exist_ok=True)
    result_json = cfg_dir / "result.json"
    ckpt_path = cfg_dir / "ckpt.pt"

    if resume and result_json.exists():
        print(f"[{run_id}] already finished — loading result.json")
        return json.load(open(result_json))

    num_gen = cfg["num_generations_stage2"]

    # initialize policies from Stage 1 template
    prey_policy_s2 = ModularPolicy(features=6).to(device)
    pred_policy_s2 = ModularPolicy(features=5).to(device)
    prey_policy_s2.load_state_dict(copy.deepcopy(prey_policy_tmpl.state_dict()))
    pred_policy_s2.load_state_dict(copy.deepcopy(pred_policy_tmpl.state_dict()))

    # joint discriminator head + optimizer (encoders frozen)
    reset_joint_head(joint_disc)
    assert_encoders_intact()
    optim_joint = torch.optim.RMSprop(
        [p for p in joint_disc.parameters() if p.requires_grad],
        lr=cfg["lr_joint_disc"], alpha=0.99, eps=1e-8
    )

    # EMA for policies
    ema_kw = dict(
        beta=cfg["ema_beta"],
        update_after_step=10,
        update_every=cfg["ema_update_every"],
        allow_different_devices=True
    )
    ema_prey = EMA(prey_policy_s2, **ema_kw)
    ema_pred = EMA(pred_policy_s2, **ema_kw)

    lr_prey, lr_pred = cfg["lr_prey_policy_s2"], cfg["lr_pred_policy_s2"]
    sigma_prey, sigma_pred = cfg["sigma_prey_s2"], cfg["sigma_pred_s2"]

    metrics_list, metrics_list_16 = [], []
    policy_metrics_list, disc_metrics_list = [], []

    best_prey, best_pred = float("inf"), float("inf")
    best_prey_gen, best_pred_gen = -1, -1
    best_prey_state = {k: v.detach().clone() for k, v in ema_prey.ema_model.state_dict().items()}
    best_pred_state = {k: v.detach().clone() for k, v in ema_pred.ema_model.state_dict().items()}
    stale_evals = 0
    start_gen = 0

    # resume from checkpoint if available
    if resume and ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location=device)
        prey_policy_s2.load_state_dict(ck["prey_policy"])
        pred_policy_s2.load_state_dict(ck["pred_policy"])
        ema_prey.load_state_dict(ck["ema_prey"])
        ema_pred.load_state_dict(ck["ema_pred"])
        joint_disc.load_state_dict(ck["joint_disc"])
        optim_joint.load_state_dict(ck["optim_joint"])

        lr_prey, lr_pred = ck["lr_prey"], ck["lr_pred"]
        sigma_prey, sigma_pred = ck["sigma_prey"], ck["sigma_pred"]

        metrics_list = ck["metrics_list"]
        metrics_list_16 = ck["metrics_list_16"]
        policy_metrics_list = ck["policy_metrics_list"]
        disc_metrics_list = ck["disc_metrics_list"]

        best_prey, best_pred = ck["best_prey"], ck["best_pred"]
        best_prey_gen, best_pred_gen = ck["best_prey_gen"], ck["best_pred_gen"]
        best_prey_state = ck["best_prey_state"]
        best_pred_state = ck["best_pred_state"]
        stale_evals = ck["stale_evals"]
        start_gen = ck["gen"] + 1
        print(f"[{run_id}] resuming at generation {start_gen}")

    t0 = time.time()

    for gen in range(start_gen, num_gen):

        # choose 16 or 32 prey condition
        n_prey = random.choice([16, 32])
        if n_prey == 16:
            cond_pool, cond_exp_pred, cond_exp_prey = (
                init_pool_16, exp_pred_tensor_16, exp_prey_tensor_16
            )
        else:
            cond_pool, cond_exp_pred, cond_exp_prey = (
                init_pool_32, exp_pred_tensor_32, exp_prey_tensor_32
            )

        # rollout with EMA policies
        gen_pred_tensor, gen_prey_tensor = run_env_vectorized(
            prey_policy=ema_prey.ema_model.to(device),
            pred_policy=ema_pred.ema_model.to(device),
            n_prey=n_prey, n_pred=1, max_steps=100, init_pool=cond_pool,
            area_height=env_settings[0], area_width=env_settings[1],
            prey_speed=env_settings[2], pred_speed=env_settings[3],
            step_size=env_settings[4], max_turn=env_settings[5],
            max_speed_norm=env_settings[7]
        )
        gen_pred_tensor, gen_prey_tensor = pad_rollout_tensors(
            gen_pred_tensor, gen_prey_tensor, max_prey=32
        )

        # discriminator schedule
        pause_period = cfg.get("disc_pause_period")
        pause_len = cfg.get("disc_pause_len", 2)
        in_pause = (
            (gen % pause_period) >= (pause_period - pause_len)
            if pause_period else False
        )
        warmup = cfg.get("disc_warmup_gens", 0)
        every = cfg.get("disc_update_every", 1)
        scheduled = (gen < warmup) or ((gen - warmup) % every == 0)
        skip_disc = in_pause or (not scheduled)

        if not skip_disc:
            for _ in range(cfg["joint_dis_balance"]):
                expert_pred_batch, e_idx = sample_data(
                    cond_exp_pred, batch_size=20, window_len=10
                )
                expert_prey_batch, _ = sample_data(
                    cond_exp_prey, batch_size=20, window_len=10, idx=e_idx
                )
                policy_pred_batch, p_idx = sample_data(
                    gen_pred_tensor, batch_size=20, window_len=10
                )
                policy_prey_batch, _ = sample_data(
                    gen_prey_tensor, batch_size=20, window_len=10, idx=p_idx
                )

                dis_metric = joint_disc.update(
                    expert_pred_batch.to(device),
                    expert_prey_batch.to(device),
                    policy_pred_batch.to(device),
                    policy_prey_batch.to(device),
                    optim_joint,
                    lambda_gp=cfg["lambda_gp_joint"],
                    noise=cfg["joint_noise"],
                    generation=gen,
                    num_generations=num_gen
                )
        else:
            dis_metric = {
                "dis_loss": float("nan"),
                "dis_loss_gp": float("nan"),
                "grad_penalty": float("nan"),
                "expert_score_mean": float("nan"),
                "policy_score_mean": float("nan"),
            }
        disc_metrics_list.append(dis_metric)

        # ES policy updates
        init_pos = init_positions(
            cond_pool, batch=num_perturbations, mode="dual",
            area_width=env_settings[1], area_height=env_settings[0]
        )

        pin_pred = optimize_es(
            pred_policy=pred_policy_s2, prey_policy=prey_policy_s2,
            role="pred", module="pairwise", mode=pred_update_mode,
            discriminator=joint_disc, lr=lr_pred, sigma=sigma_pred,
            num_perturbations=num_perturbations, init_pos=init_pos,
            settings_batch_env=env_settings, n_prey=n_prey,
            theta_norm_ref=THETA_REF[("pred", "pairwise")]
        )
        an_pred = optimize_es(
            pred_policy=pred_policy_s2, prey_policy=prey_policy_s2,
            role="pred", module="attention", mode=pred_update_mode,
            discriminator=joint_disc, lr=lr_pred, sigma=sigma_pred,
            num_perturbations=num_perturbations, init_pos=init_pos,
            settings_batch_env=env_settings, n_prey=n_prey,
            theta_norm_ref=THETA_REF[("pred", "attention")]
        )
        ema_pred.update()

        pin_prey = an_prey = None
        if not cfg.get("prey_frozen", False):
            pin_prey = optimize_es(
                pred_policy=pred_policy_s2, prey_policy=prey_policy_s2,
                role="prey", module="pairwise", mode=prey_update_mode,
                discriminator=joint_disc, lr=lr_prey, sigma=sigma_prey,
                num_perturbations=num_perturbations, init_pos=init_pos,
                settings_batch_env=env_settings, n_prey=n_prey,
                theta_norm_ref=THETA_REF[("prey", "pairwise")]
            )
            an_prey = optimize_es(
                pred_policy=pred_policy_s2, prey_policy=prey_policy_s2,
                role="prey", module="attention", mode=prey_update_mode,
                discriminator=joint_disc, lr=lr_prey, sigma=sigma_prey,
                num_perturbations=num_perturbations, init_pos=init_pos,
                settings_batch_env=env_settings, n_prey=n_prey,
                theta_norm_ref=THETA_REF[("prey", "attention")]
            )
            ema_prey.update()

        policy_metrics_list.append({
            "predator": (pin_pred, an_pred),
            "prey": (pin_prey, an_prey),
            "n_prey": n_prey
        })

        lr_pred *= cfg["gamma_s2"]
        lr_prey *= cfg["gamma_s2"]
        sigma_pred *= cfg["gamma_s2"]
        sigma_prey *= cfg["gamma_s2"]

        # evaluation on EMA models
        if gen % performance_eval == 0:
            m32 = calculate_metrics(
                pred_policy=ema_pred.ema_model,
                prey_policy=ema_prey.ema_model,
                prey_encoder=prey_encoder,
                pred_encoder=pred_encoder,
                exp_prey_tensor=exp_prey_tensor_32,
                exp_pred_tensor=exp_pred_tensor_32,
                prey_mmd_loss=prey_mmd_loss,
                pred_mmd_loss=pred_mmd_loss,
                sinkhorn_loss=sinkhorn_loss,
                init_pool=init_pool_32,
                n_prey=32,
                env_settings=env_settings,
                device="cuda"
            )
            m16 = calculate_metrics(
                pred_policy=ema_pred.ema_model,
                prey_policy=ema_prey.ema_model,
                prey_encoder=prey_encoder,
                pred_encoder=pred_encoder,
                exp_prey_tensor=exp_prey_tensor_16,
                exp_pred_tensor=exp_pred_tensor_16,
                prey_mmd_loss=prey_mmd_loss,
                pred_mmd_loss=pred_mmd_loss,
                sinkhorn_loss=sinkhorn_loss,
                init_pool=init_pool_16,
                n_prey=16,
                env_settings=env_settings,
                device="cuda"
            )
            metrics_list.append({"gen": gen, **m32})
            metrics_list_16.append({"gen": gen, **m16})

            # inline combined score
            prey_score = float(
                m32["mmd_prey_mean"] / MMD_PREY
                + m32["sinkhorn_prey_mean"] / SNK_PREY
                + m16["mmd_prey_mean"] / MMD_PREY16
                + m16["sinkhorn_prey_mean"] / SNK_PREY16
            )
            pred_score = float(
                m32["mmd_pred_mean"] / MMD_PRED
                + m32["sinkhorn_pred_mean"] / SNK_PRED
                + m16["mmd_pred_mean"] / MMD_PRED16
                + m16["sinkhorn_pred_mean"] / SNK_PRED16
            )

            improved = False
            if prey_score < best_prey:
                best_prey = prey_score
                best_prey_gen = gen
                improved = True
                best_prey_state = {
                    k: v.detach().clone()
                    for k, v in ema_prey.ema_model.state_dict().items()
                }
            if pred_score < best_pred:
                best_pred = pred_score
                best_pred_gen = gen
                improved = True
                best_pred_state = {
                    k: v.detach().clone()
                    for k, v in ema_pred.ema_model.state_dict().items()
                }

            stale_evals = 0 if improved else stale_evals + 1
            print(
                f"  [{run_id}] gen {gen:4d} | "
                f"prey {prey_score:7.3f} (best {best_prey:7.3f}) | "
                f"pred {pred_score:7.3f} (best {best_pred:7.3f}) | "
                f"stale {stale_evals}"
            )

            patience = cfg.get("patience")
            if patience is not None and stale_evals >= patience:
                print(
                    f"  [{run_id}] early stop at gen {gen} "
                    f"({patience} evals without improvement)"
                )
                break

        # checkpointing
        if (gen + 1) % ckpt_every == 0:
            torch.save(
                {
                    "gen": gen,
                    "prey_policy": prey_policy_s2.state_dict(),
                    "pred_policy": pred_policy_s2.state_dict(),
                    "ema_prey": ema_prey.state_dict(),
                    "ema_pred": ema_pred.state_dict(),
                    "joint_disc": joint_disc.state_dict(),
                    "optim_joint": optim_joint.state_dict(),
                    "lr_prey": lr_prey,
                    "lr_pred": lr_pred,
                    "sigma_prey": sigma_prey,
                    "sigma_pred": sigma_pred,
                    "metrics_list": metrics_list,
                    "metrics_list_16": metrics_list_16,
                    "policy_metrics_list": policy_metrics_list,
                    "disc_metrics_list": disc_metrics_list,
                    "best_prey": best_prey,
                    "best_pred": best_pred,
                    "best_prey_gen": best_prey_gen,
                    "best_pred_gen": best_pred_gen,
                    "best_prey_state": best_prey_state,
                    "best_pred_state": best_pred_state,
                    "stale_evals": stale_evals,
                    "cfg": cfg,
                },
                ckpt_path
            )

    # save best policies and results
    torch.save(best_prey_state, cfg_dir / "best_prey_policy.pth")
    torch.save(best_pred_state, cfg_dir / "best_pred_policy.pth")

    result = {
        "run_id": run_id,
        "cfg": cfg,
        "best_prey_score": best_prey,
        "best_pred_score": best_pred,
        "best_prey_gen": best_prey_gen,
        "best_pred_gen": best_pred_gen,
        "metrics": metrics_list,
        "metrics_16": metrics_list_16,
        "disc_metrics": disc_metrics_list,
        "minutes": round((time.time() - t0) / 60, 1),
    }
    json.dump(result, open(result_json, "w"), indent=2, default=float)
    print(
        f"[{run_id}] done in {result['minutes']} min — "
        f"prey {best_prey:.3f} @gen {best_prey_gen} | "
        f"pred {best_pred:.3f} @gen {best_pred_gen}"
    )
    return result

# run sweeps
results_p1 = [run_one_config(cfg, SWEEP_DIR) for cfg in phase1]

In [ ]:
# configurations

BASE = {
    "num_generations_stage2": 300,
    "lr_prey_policy_s2":  5e-5, # Stage 1: 2e-4
    "lr_pred_policy_s2":  1e-5, # Stage 1: 1e-4
    "sigma_prey_s2":      0.05, # Stage 1: 0.1
    "sigma_pred_s2":      0.07, # Stage 1: 0.08
    "gamma_s2":           0.997,
    "lr_joint_disc":      3e-4, # between prey 5e-4 and pred 2e-4
    "joint_dis_balance":  2, # matches Stage 1
    "lambda_gp_joint":    6, # between prey 5 and pred 10
    "joint_noise":        0.005, # matches Stage 1
    "disc_warmup_gens":   0,
    "disc_update_every":  1,
    "patience":           None, # off, so every config gets the same budget
    "prey_frozen":        False,
    "ema_beta":           0.99, # ~100-update horizon, matched to 300 gens
    "ema_update_every":   1,
}

def cfg(run_id, **overrides):
    return {"run_id": run_id, **BASE, **overrides}

# I decided to stick with the following configurations because in early tests of 25 generation sweeps, they seemed to do the best
phase1 = [
    cfg("00_baseline"),
    cfg("01_dis_balance_1",  joint_dis_balance=1), # does the joint disc need fewer steps than two per-role discs?
    cfg("02_lambda_gp_10",   lambda_gp_joint=10), # stronger Lipschitz (pred Stage 1 value)
    cfg("03_soft_gp",        lambda_gp_joint=3), # is the GP too tight to be informative?
    cfg("04_high_noise",     joint_noise=0.025), # does more input noise stop joint-disc overconfidence?
    cfg("05_higher_pred_lr", lr_pred_policy_s2=3e-5), # TTUR (Heusel et al. 2017)
    cfg("06_prey_frozen",    prey_frozen=True), # curriculum (Bengio et al. 2009)
    cfg("07_disc_every_2",   disc_update_every=2, disc_warmup_gens=50),
]

results_p1 = run_sweep(phase1, SWEEP_DIR)

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


  [00_baseline] gen    0 | prey  24.096 (best  24.096) | pred  15.165 (best  15.165) | stale 0
  [00_baseline] gen    5 | prey  28.770 (best  24.096) | pred  17.209 (best  15.165) | stale 1
  [00_baseline] gen   10 | prey  27.182 (best  24.096) | pred  18.832 (best  15.165) | stale 2
  [00_baseline] gen   15 | prey  25.643 (best  24.096) | pred  18.473 (best  15.165) | stale 3
  [00_baseline] gen   20 | prey  29.573 (best  24.096) | pred  19.912 (best  15.165) | stale 4
  [00_baseline] gen   25 | prey  29.668 (best  24.096) | pred  20.313 (best  15.165) | stale 5
  [00_baseline] gen   30 | prey  33.825 (best  24.096) | pred  22.891 (best  15.165) | stale 6
  [00_baseline] gen   35 | prey  32.398 (best  24.096) | pred  21.574 (best  15.165) | stale 7
  [00_baseline] gen   40 | prey  35.187 (best  24.096) | pred  22.864 (best  15.165) | stale 8
  [00_baseline] gen   45 | prey  35.859 (best  24.096) | pred  24.034 (best  15.165) | stale 9
  [00_baseline] gen   50 | prey  34.906 (best  24.

In [ ]:
# summary table
def summarize(results):
    rows = []
    for r in results:
        m32 = min(r["metrics"],    key=lambda m: m["sinkhorn_pred_mean"])
        m16 = min(r["metrics_16"], key=lambda m: m["sinkhorn_pred_mean"])
        rows.append({
            "run_id":       r["run_id"],
            "prey_score":   r["best_prey_score"],
            "pred_score":   r["best_pred_score"],
            "prey_gen":     r["best_prey_gen"],
            "pred_gen":     r["best_pred_gen"],
            "snk_prey_32":  m32["sinkhorn_prey_mean"],
            "snk_pred_32":  m32["sinkhorn_pred_mean"],
            "mmd_prey_32":  m32["mmd_prey_mean"],
            "mmd_pred_32":  m32["mmd_pred_mean"],
            "snk_prey_16":  m16["sinkhorn_prey_mean"],
            "snk_pred_16":  m16["sinkhorn_pred_mean"],
            "min":          r["minutes"],
        })
    df = pd.DataFrame(rows).sort_values("pred_score").reset_index(drop=True)
    print(df.to_string(index=False))
    return df

df_sweep = summarize(results_p1)
df_sweep.to_csv(SWEEP_DIR / "sweep_summary.csv", index=False)

print(f"\nexpert 32  prey snk {SNK_PREY:.3e}  pred snk {SNK_PRED:.3e}  "
      f"prey mmd {MMD_PREY:.4f}  pred mmd {MMD_PRED:.4f}")
print(f"expert 16  prey snk {SNK_PREY16:.3e}  pred snk {SNK_PRED16:.3e}  "
      f"prey mmd {MMD_PREY16:.4f}  pred mmd {MMD_PRED16:.4f}")

           run_id  prey_score  pred_score  prey_gen  pred_gen  snk_prey_32  snk_pred_32  mmd_prey_32  mmd_pred_32  snk_prey_16  snk_pred_16  min
  07_disc_every_2   24.136658   13.346257       295       190     0.042381     0.032705     0.719115     2.133282     0.004251     0.019517 27.8
      00_baseline   23.589970   13.423072       145       160     0.039938     0.033162     0.661078     1.941659     0.004358     0.020375 27.9
05_higher_pred_lr   24.285147   13.650432       205       115     0.036601     0.030943     1.363811     2.784822     0.004134     0.018640 27.8
 01_dis_balance_1   23.516922   14.351874       140       155     0.041774     0.034237     0.678615     2.107624     0.004248     0.021153 27.7
       03_soft_gp   26.137707   14.437566       110       105     0.035464     0.030083     1.040203     2.743493     0.004039     0.019002 28.0
   06_prey_frozen   25.498039   14.580793       195        35     0.041490     0.034370     0.727448     1.820070     0.004313    